# 🌧️ Погода для почвоведа
## Где взять, как подготовить и как использовать метеоданные

**Школа молодых почвоведов — Цифровой трек, 2026**

---

Этот ноутбук — рабочий инструмент. В конце занятия вы сможете взять его с собой и запустить для **любой своей точки**.

Мы работаем с данными для района **Тверь / ЦДП** — 41 год наблюдений (1980–2020).

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  ЗАПУСТИТЕ ЭТУ ЯЧЕЙКУ ПЕРВОЙ — скачивает данные    ║
# ╚══════════════════════════════════════════════════════╝
!wget -q https://raw.githubusercontent.com/margo-u/shmp-2026-meteo/main/tver_meteo_final.csv
print("✅ Данные загружены! Можно идти дальше.")

In [ ]:
!pip install pandas matplotlib seaborn scipy --quiet
print('✅ Всё готово!')

---
## 1. Загружаем данные

В файле два источника:
- **Станция** — реальные наземные наблюдения (метеостанция Тверь)
- **ERA5** — реанализ ECMWF (глобальная модель, сетка ~31 км)

Оба — в одной таблице, уже совмещены.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
import seaborn as sns
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# --- ЗАГРУЗКА ---
# В Google Colab раскомментируйте две строки ниже:
# from google.colab import files
# files.upload()  # загрузите tver_meteo_final.csv

df = pd.read_csv('tver_meteo_final.csv', parse_dates=['date'])
df = df.set_index('date')
df['month'] = df.index.month
df['year']  = df.index.year
df['doy']   = df.index.dayofyear

MONTHS_RU = ['Янв','Фев','Мар','Апр','Май','Июн','Июл','Авг','Сен','Окт','Ноя','Дек']

print(f'📅 Период: {df.index.min().date()} — {df.index.max().date()}')
print(f'📊 Строк: {len(df):,} (дней, {df["year"].nunique()} лет)')
print(f'\n📋 Переменные:')
print('  Станция : Tmean, Tmin, Tmax (°C) | Prcp (мм/сут) | RH_mean_filled (%) | Wind_mean_filled (м/с)')
print('  ERA5    : ERA5_Tmean, ERA5_Tmin, ERA5_Tmax (°C) | ERA5_Prcp (мм/сут)')
print('  Прочее  : ssrd_MJ (суммарная радиация) | Rn_MJ (чистая радиация)')

In [ ]:
# Первый взгляд на таблицу
df[['Tmean','Tmin','Tmax','Prcp','ERA5_Tmean','ERA5_Prcp','RH_mean_filled','Wind_mean_filled']].head(10)

---
## 2. Сезонный ход температуры и осадков

**❓ Вопрос:** Глядя только на этот график — что можно сказать об этой территории?

Подумайте: форма кривых, амплитуды, распределение осадков по сезонам.

In [ ]:
n_years = df['year'].nunique()
monthly = df.groupby('month').agg(
    T_st   = ('Tmean',    'mean'),
    T_era5 = ('ERA5_Tmean','mean'),
    P_st   = ('Prcp',     lambda x: x.sum() / n_years),
    P_era5 = ('ERA5_Prcp',lambda x: x.sum() / n_years)
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Температура
ax = axes[0]
ax.plot(monthly['month'], monthly['T_st'],   'o-', color='#e74c3c', lw=2.5, ms=8, label='Станция')
ax.plot(monthly['month'], monthly['T_era5'], 's--',color='#3498db', lw=2,   ms=7, label='ERA5')
ax.axhline(0,  color='gray', lw=1, ls=':')
ax.axhline(5,  color='#27ae60', lw=1, ls=':', alpha=0.7)
ax.fill_between(monthly['month'], 0, monthly['T_st'],
                where=monthly['T_st']>0, alpha=0.12, color='#e74c3c')
ax.fill_between(monthly['month'], 0, monthly['T_st'],
                where=monthly['T_st']<0, alpha=0.12, color='#3498db')
ax.set_xticks(range(1,13)); ax.set_xticklabels(MONTHS_RU)
ax.set_ylabel('Температура (°C)'); ax.set_title('Среднемесячная температура (1980–2020)')
ax.legend()

# Осадки
ax = axes[1]
x = np.arange(12); w = 0.35
ax.bar(x-w/2, monthly['P_st'],   w, color='#2ecc71', alpha=0.85, label='Станция')
ax.bar(x+w/2, monthly['P_era5'], w, color='#9b59b6', alpha=0.85, label='ERA5')
ax.set_xticks(x); ax.set_xticklabels(MONTHS_RU)
ax.set_ylabel('Осадки (мм/месяц)'); ax.set_title('Среднемесячные осадки (1980–2020)')
ax.legend()

plt.tight_layout()
plt.savefig('01_seasonal.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 01_seasonal.png')

---
## 3. Климадиаграмма Вальтера–Лита

Классика климатологии и почвоведения. Левая шкала — температура (°C), правая — осадки (мм), но **в масштабе 1°C = 2 мм**. Если кривая осадков выше кривой температуры — период влажный (синий), ниже — сухой (жёлтый).

In [ ]:
fig, ax1 = plt.subplots(figsize=(11, 5))

T = monthly['T_st'].values
P = monthly['P_st'].values
P_scaled = P / 2  # приводим к шкале температуры
m = np.arange(1, 13)

# Фон: влажные/сухие периоды
for i in range(len(m)-1):
    x_seg = [m[i], m[i+1]]
    t_seg = [T[i], T[i+1]]
    p_seg = [P_scaled[i], P_scaled[i+1]]
    if P_scaled[i] >= T[i]:
        ax1.fill_between(x_seg, t_seg, p_seg, alpha=0.25, color='#3498db')
    else:
        ax1.fill_between(x_seg, t_seg, p_seg, alpha=0.25, color='#f39c12')

ax1.plot(m, T,        'o-', color='#e74c3c', lw=2.5, ms=8, label='Температура')
ax1.plot(m, P_scaled, 's-', color='#3498db', lw=2.5, ms=8, label='Осадки (P/2)')
ax1.axhline(0, color='gray', lw=1)

ax1.set_xticks(m); ax1.set_xticklabels(MONTHS_RU)
ax1.set_ylabel('Температура (°C)', color='#e74c3c')
ax1.tick_params(axis='y', labelcolor='#e74c3c')

ax2 = ax1.twinx()
ax2.set_ylim(ax1.get_ylim()[0]*2, ax1.get_ylim()[1]*2)
ax2.set_ylabel('Осадки (мм)', color='#3498db')
ax2.tick_params(axis='y', labelcolor='#3498db')

patch_w = mpatches.Patch(color='#3498db', alpha=0.4, label='Влажный период')
patch_d = mpatches.Patch(color='#f39c12', alpha=0.4, label='Сухой период')
ax1.legend(handles=[plt.Line2D([],[],color='#e74c3c',lw=2,marker='o',label='Температура'),
                    plt.Line2D([],[],color='#3498db',lw=2,marker='s',label='Осадки (P/2)'),
                    patch_w, patch_d], loc='lower right', fontsize=10)

T_ann = df['Tmean'].mean()
P_ann = df['Prcp'].sum() / n_years
ax1.set_title(f'Климадиаграмма Вальтера–Лита: Тверь\n'
              f'T̄ = {T_ann:.1f}°C   P̄ = {P_ann:.0f} мм/год')

plt.tight_layout()
plt.savefig('02_walter_lieth.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 02_walter_lieth.png')
print(f'\n📌 Вывод: все месяцы влажные (P/2 > T) — типично для лесной зоны умеренного климата')

---
## 4. Агроклиматические индексы

### 4.1 Гидротермический коэффициент Селянинова (ГТК)

$$\text{ГТК} = \frac{\sum P}{0.1 \cdot \sum T^+}$$

| ГТК | Увлажнение |
|-----|------------|
| < 0.5 | Очень сухо |
| 0.5–1.0 | Сухо |
| 1.0–1.5 | Недостаточное |
| **1.5–2.0** | **Достаточное** |
| > 2.0 | Избыточное |

In [ ]:
veg = df[df['month'].isin([5,6,7,8,9])].copy()
veg['T_pos'] = veg['Tmean'].clip(lower=0)

gtk = veg.groupby('year').apply(
    lambda x: x['Prcp'].sum() / (0.1 * x['T_pos'].sum()),
    include_groups=False
).reset_index()
gtk.columns = ['year','GTK']

colors = ['#e74c3c' if v<1.0 else '#f39c12' if v<1.5 else '#27ae60' if v<=2.0 else '#2980b9'
          for v in gtk['GTK']]

fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(gtk['year'], gtk['GTK'], color=colors, alpha=0.85, edgecolor='white', lw=0.5)
ax.axhline(1.5, color='#27ae60', lw=2, ls='--', label='1.5 — нижняя граница достаточного')
ax.axhline(2.0, color='#2980b9', lw=2, ls='--', label='2.0 — избыточное')
ax.axhline(1.0, color='#e74c3c', lw=1.5, ls=':', label='1.0')
ax.axhline(gtk['GTK'].mean(), color='black', lw=1.5, ls='-',
           label=f'Среднее = {gtk["GTK"].mean():.2f}')
ax.set_xlabel('Год'); ax.set_ylabel('ГТК')
ax.set_title('Гидротермический коэффициент Селянинова (май–сентябрь), Тверь')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('03_gtk.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 03_gtk.png')
print(f"\nГТК среднее : {gtk['GTK'].mean():.2f}")
print(f"Мин         : {gtk['GTK'].min():.2f} ({gtk.loc[gtk['GTK'].idxmin(),'year']}) — засуха")
print(f"Макс        : {gtk['GTK'].max():.2f} ({gtk.loc[gtk['GTK'].idxmax(),'year']}) — переувлажнение")

### 4.2 Суммы активных температур (САТ)

In [ ]:
df['T5']  = df['Tmean'].where(df['Tmean'] > 5,  0).clip(lower=0)
df['T10'] = df['Tmean'].where(df['Tmean'] > 10, 0).clip(lower=0)

sat = df.groupby('year').agg(SAT5=('T5','sum'), SAT10=('T10','sum')).reset_index()

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(sat['year'], sat['SAT5'],  'o-', color='#e67e22', lw=2, ms=5, label='САТ > 5°C')
ax.plot(sat['year'], sat['SAT10'], 's-', color='#e74c3c', lw=2, ms=5, label='САТ > 10°C')

for col, color, label in [('SAT5','#e67e22','> 5°C'),('SAT10','#e74c3c','> 10°C')]:
    sl, ic, r, p, _ = stats.linregress(sat['year'], sat[col])
    ax.plot(sat['year'], ic + sl*sat['year'], '--', color=color, alpha=0.6, lw=1.5)
    print(f'САТ {label}: тренд {sl*10:+.1f} °С·сут/10 лет  (p={p:.3f})')

ax.set_xlabel('Год'); ax.set_ylabel('Сумма активных температур (°С·сут)')
ax.set_title('Суммы активных температур, Тверь')
ax.legend()
plt.tight_layout()
plt.savefig('04_sat.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 04_sat.png')

---
## 5. Безморозный период

Период между последним весенним и первым осенним заморозком (Tmin < 0°C). Один из важнейших показателей для оценки почвообразования и сельского хозяйства.

In [ ]:
records = []
for yr in sorted(df['year'].unique()):
    y = df[df['year'] == yr]
    spring = y[(y.index.month < 7) & (y['Tmin'] < 0)]
    fall   = y[(y.index.month >= 7) & (y['Tmin'] < 0)]
    if len(spring) > 0 and len(fall) > 0:
        records.append({
            'year'  : yr,
            'ls_doy': spring.index[-1].dayofyear,   # последний весенний заморозок
            'ff_doy': fall.index[0].dayofyear,       # первый осенний заморозок
            'ffp'   : fall.index[0].dayofyear - spring.index[-1].dayofyear
        })

ffp = pd.DataFrame(records)

fig, axes = plt.subplots(2, 1, figsize=(13, 7))

# Длина безморозного периода
ax = axes[0]
ax.bar(ffp['year'], ffp['ffp'], color='#27ae60', alpha=0.8)
sl, ic, r, p, _ = stats.linregress(ffp['year'], ffp['ffp'])
ax.plot(ffp['year'], ic + sl*ffp['year'], 'k--', lw=2,
        label=f'Тренд: {sl*10:+.1f} дней/10 лет (p={p:.3f})')
ax.axhline(ffp['ffp'].mean(), color='gray', lw=1, ls=':',
           label=f'Среднее: {ffp["ffp"].mean():.0f} дней')
ax.set_ylabel('Длина безморозного периода (дней)')
ax.set_title('Безморозный период, Тверь')
ax.legend()

# Даты начала и конца
ax = axes[1]
ax.fill_between(ffp['year'], ffp['ls_doy'], ffp['ff_doy'],
                alpha=0.35, color='#27ae60', label='Безморозный период')
ax.plot(ffp['year'], ffp['ls_doy'], 'v', color='#3498db', ms=7,
        label='Последний весенний заморозок')
ax.plot(ffp['year'], ffp['ff_doy'], '^', color='#e74c3c', ms=7,
        label='Первый осенний заморозок')

# Метки по оси Y → даты
ticks_doy = [91, 121, 152, 182, 213, 244, 274, 305]
ticks_lbl = ['1 апр','1 май','1 июн','1 июл','1 авг','1 сен','1 окт','1 ноя']
ax.set_yticks(ticks_doy); ax.set_yticklabels(ticks_lbl)
ax.set_ylabel('Дата'); ax.set_xlabel('Год')
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('05_frost_free.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 05_frost_free.png')
print(f"\nСредний безморозный период: {ffp['ffp'].mean():.0f} дней")
print(f"Самый длинный: {ffp['ffp'].max()} дней ({ffp.loc[ffp['ffp'].idxmax(),'year']})")
print(f"Самый короткий: {ffp['ffp'].min()} дней ({ffp.loc[ffp['ffp'].idxmin(),'year']})")

---
## 6. Радиационный баланс

В файле есть два поля радиации:
- **ssrd_MJ** — суммарная солнечная радиация, приходящая на поверхность
- **Rn_MJ** — чистая (результирующая) радиация = приход − расход

Чистая радиация — двигатель испарения и нагрева почвы.

In [ ]:
rad_monthly = df.groupby('month').agg(
    ssrd=('ssrd_MJ','mean'),
    Rn  =('Rn_MJ',  'mean'),
    T   =('Tmean',  'mean')
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Сезонный ход радиации
ax = axes[0]
ax.bar(rad_monthly['month']-0.2, rad_monthly['ssrd'], 0.4,
       color='#f39c12', alpha=0.85, label='Суммарная (ssrd)')
ax.bar(rad_monthly['month']+0.2, rad_monthly['Rn'],   0.4,
       color='#e74c3c', alpha=0.85, label='Чистая (Rn)')
ax.axhline(0, color='gray', lw=1)
ax.set_xticks(range(1,13)); ax.set_xticklabels(MONTHS_RU)
ax.set_ylabel('МДж/м²/сут'); ax.set_title('Среднесуточная радиация по месяцам')
ax.legend()

# Годовые суммы
rad_annual = df.groupby('year').agg(ssrd=('ssrd_MJ','sum'), Rn=('Rn_MJ','sum')).reset_index()
ax = axes[1]
ax.plot(rad_annual['year'], rad_annual['ssrd'], 'o-', color='#f39c12', lw=2, ms=5, label='Суммарная (ssrd)')
ax.plot(rad_annual['year'], rad_annual['Rn'],   's-', color='#e74c3c', lw=2, ms=5, label='Чистая (Rn)')
ax.set_xlabel('Год'); ax.set_ylabel('МДж/м²/год')
ax.set_title('Годовые суммы радиации')
ax.legend()

plt.tight_layout()
plt.savefig('06_radiation.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 06_radiation.png')
print(f"\nСредняя годовая сумма суммарной радиации : {rad_annual['ssrd'].mean():.0f} МДж/м²")
print(f"Средняя годовая сумма чистой радиации    : {rad_annual['Rn'].mean():.0f} МДж/м²")
print(f"Доля чистой от суммарной                 : {rad_annual['Rn'].mean()/rad_annual['ssrd'].mean()*100:.0f}%")

---
## 7. Влажность и ветер

In [ ]:
rh_wind = df.groupby('month').agg(
    RH  =('RH_mean_filled',   'mean'),
    Wind=('Wind_mean_filled',  'mean')
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
ax.bar(rh_wind['month'], rh_wind['RH'], color='#3498db', alpha=0.8)
ax.set_xticks(range(1,13)); ax.set_xticklabels(MONTHS_RU)
ax.set_ylabel('Относительная влажность (%)'); ax.set_ylim(0, 100)
ax.set_title('Среднемесячная относительная влажность воздуха')
ax.axhline(80, color='red', lw=1, ls='--', alpha=0.6, label='80%')
ax.legend()

ax = axes[1]
ax.bar(rh_wind['month'], rh_wind['Wind'], color='#95a5a6', alpha=0.85)
ax.set_xticks(range(1,13)); ax.set_xticklabels(MONTHS_RU)
ax.set_ylabel('Скорость ветра (м/с)')
ax.set_title('Среднемесячная скорость ветра')

plt.tight_layout()
plt.savefig('07_rh_wind.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 07_rh_wind.png')
print(f"\nСредняя относительная влажность: {df['RH_mean_filled'].mean():.1f}%")
print(f"Средняя скорость ветра          : {df['Wind_mean_filled'].mean():.2f} м/с")
print(f"Макс. скорость ветра            : {df['Wind_mean_filled'].max():.1f} м/с")

---
## 8. Масштаб имеет значение

Один и тот же вопрос «как меняется температура» — три разных ответа в зависимости от масштаба.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 11))

# 1. Суточный ряд (один год)
ax = axes[0]
year_show = 2010
d = df[df['year'] == year_show]
ax.fill_between(d.index, d['Tmin'], d['Tmax'], alpha=0.25, color='#e74c3c', label='Tmin–Tmax')
ax.plot(d.index, d['Tmean'], color='#c0392b', lw=1.5, label='Tmean')
ax.axhline(0, color='gray', lw=1)
ax.set_title(f'Суточный ход температуры ({year_show})')
ax.set_ylabel('T (°C)'); ax.legend(loc='upper left')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))

# 2. Среднегодовая T
ax = axes[1]
t_ann = df.groupby('year')['Tmean'].mean()
ax.bar(t_ann.index, t_ann.values, color='#e74c3c', alpha=0.7)
sl, ic, r, p, _ = stats.linregress(t_ann.index, t_ann.values)
ax.plot(t_ann.index, ic + sl*t_ann.index, 'k--', lw=2,
        label=f'Тренд: {sl*10:+.2f}°C/10 лет (p={p:.3f})')
ax.set_ylabel('T (°C)'); ax.set_title('Среднегодовая температура (1980–2020)')
ax.legend()

# 3. Десятилетний сезонный ход
ax = axes[2]
df['decade'] = (df['year'] // 10) * 10
t_dec = df.groupby(['decade','month'])['Tmean'].mean().reset_index()
palette = ['#3498db','#2ecc71','#f39c12','#e74c3c']
for i, dec in enumerate(sorted(t_dec['decade'].unique())):
    sub = t_dec[t_dec['decade']==dec]
    ax.plot(sub['month'], sub['Tmean'], 'o-',
            color=palette[i], lw=2.5, ms=7, label=f'{dec}–{dec+9}')
ax.axhline(0, color='gray', lw=1)
ax.set_xticks(range(1,13)); ax.set_xticklabels(MONTHS_RU)
ax.set_ylabel('T (°C)'); ax.set_title('Сезонный ход по десятилетиям')
ax.legend(title='Десятилетие')

plt.tight_layout()
plt.savefig('08_scales.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 08_scales.png')
print(f'\n🌡️ Потепление за 40 лет: {sl*40:+.2f}°C')

---
## 9. Станция vs ERA5

Как хорошо реанализ воспроизводит наземные наблюдения?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Температура
ax = axes[0]
sample = df[['Tmean','ERA5_Tmean']].dropna().sample(2000, random_state=42)
ax.scatter(sample['Tmean'], sample['ERA5_Tmean'], alpha=0.15, s=10, color='#3498db')
lims = [-40, 33]
ax.plot(lims, lims, 'r--', lw=1.5, label='1:1')
r, _ = stats.pearsonr(sample['Tmean'], sample['ERA5_Tmean'])
bias_t = (df['ERA5_Tmean'] - df['Tmean']).mean()
ax.set_xlabel('Tmean станция (°C)'); ax.set_ylabel('ERA5 Tmean (°C)')
ax.set_title(f'Температура: r = {r:.3f},  смещение = {bias_t:+.2f}°C')
ax.set_xlim(lims); ax.set_ylim(lims); ax.legend()

# Осадки
ax = axes[1]
sp = df[['Prcp','ERA5_Prcp']].dropna()
sp = sp[sp['Prcp'] < 25]
ax.scatter(sp['Prcp'], sp['ERA5_Prcp'], alpha=0.1, s=8, color='#27ae60')
ax.plot([0,25],[0,25], 'r--', lw=1.5, label='1:1')
r_p, _ = stats.pearsonr(sp['Prcp'], sp['ERA5_Prcp'])
bias_p = (df['ERA5_Prcp'] - df['Prcp']).mean()
ax.set_xlabel('Осадки станция (мм/сут)'); ax.set_ylabel('ERA5 осадки (мм/сут)')
ax.set_title(f'Осадки: r = {r_p:.3f},  смещение = {bias_p:+.2f} мм/сут')
ax.legend()

plt.tight_layout()
plt.savefig('09_vs_era5.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 09_vs_era5.png')
print(f'\n❓ Почему r для осадков намного ниже, чем для температуры?')

---
## 10. Экстремальные события

In [ ]:
frost  = df[df['Tmin'] < 0].groupby('year').size()
hot    = df[df['Tmax'] > 25].groupby('year').size()
heavy  = df[df['Prcp'] > 10].groupby('year').size()  # сильные осадки

fig, axes = plt.subplots(3, 1, figsize=(13, 9))

for ax, data, color, label, unit in [
    (axes[0], frost, '#3498db', 'Дней с Tmin < 0°C (морозные)', 'дней/год'),
    (axes[1], hot,   '#e74c3c', 'Дней с Tmax > 25°C (жаркие)',  'дней/год'),
    (axes[2], heavy, '#9b59b6', 'Дней с осадками > 10 мм (ливни)', 'дней/год'),
]:
    ax.bar(data.index, data.values, color=color, alpha=0.8)
    sl, ic, r, p, _ = stats.linregress(data.index, data.values)
    ax.plot(data.index, ic + sl*data.index, 'k--', lw=1.5,
            label=f'Тренд: {sl*10:+.1f} {unit}/10 лет (p={p:.3f})')
    ax.set_ylabel(unit); ax.set_title(label); ax.legend()

plt.tight_layout()
plt.savefig('10_extremes.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 10_extremes.png')
print(f"\nАбс. минимум: {df['Tmin'].min():.1f}°C ({df['Tmin'].idxmin().date()})")
print(f"Абс. максимум: {df['Tmax'].max():.1f}°C ({df['Tmax'].idxmax().date()})")
print(f"Макс. осадки за сутки: {df['Prcp'].max():.1f} мм ({df['Prcp'].idxmax().date()})")

---
## 11. Климатический портрет территории

Итоговая сводка — то, что обычно нужно почвоведу для раздела «Климат» в статье или диссертации.

In [ ]:
t_ann  = df.groupby('year')['Tmean'].mean().mean()
p_ann  = df.groupby('year')['Prcp'].sum().mean()
t_jan  = df[df['month']==1]['Tmean'].mean()
t_jul  = df[df['month']==7]['Tmean'].mean()
sat5m  = df.groupby('year')['T5'].sum().mean()
sat10m = df.groupby('year')['T10'].sum().mean()
gtk_m  = gtk['GTK'].mean()
ffp_m  = ffp['ffp'].mean()
sl_t,_,_,_,_ = stats.linregress(t_ann := df.groupby('year')['Tmean'].mean(), t_ann)

# пересчитаем тренд правильно
t_annual = df.groupby('year')['Tmean'].mean()
sl_t, ic_t, *_ = stats.linregress(t_annual.index, t_annual.values)

print('=' * 58)
print('      КЛИМАТИЧЕСКИЙ ПОРТРЕТ: ТВЕРЬ (1980–2020)')
print('=' * 58)
rows = [
    ('Среднегодовая температура',      f'{t_ann:.1f} °C'),
    ('Средняя температура января',      f'{t_jan:.1f} °C'),
    ('Средняя температура июля',        f'{t_jul:.1f} °C'),
    ('Годовая амплитуда',               f'{t_jul-t_jan:.1f} °C'),
    ('Абсолютный минимум',              f"{df['Tmin'].min():.1f} °C (янв 1987)"),
    ('Абсолютный максимум',             f"{df['Tmax'].max():.1f} °C (июл 2010)"),
    ('---',                             '---'),
    ('Среднегодовые осадки',            f'{p_ann:.0f} мм'),
    ('Макс. суточные осадки',           f"{df['Prcp'].max():.1f} мм"),
    ('---',                             '---'),
    ('Безморозный период (средний)',    f'{ffp_m:.0f} дней'),
    ('САТ > 5°C',                       f'{sat5m:.0f} °С·сут'),
    ('САТ > 10°C',                      f'{sat10m:.0f} °С·сут'),
    ('ГТК Селянинова (май–сент)',        f'{gtk_m:.2f}'),
    ('---',                             '---'),
    ('Тренд температуры',               f'{sl_t*10:+.2f} °C/10 лет'),
]
for lbl, val in rows:
    if lbl == '---': print('-' * 58)
    else: print(f'  {lbl:<38} {val}')
print('=' * 58)
print(f'\n📌 Вывод: достаточное увлажнение (ГТК={gtk_m:.2f}), умеренно')
print('   континентальный климат, зона дерново-подзолистых почв')

---
## 🏠 Ваша точка — Open-Meteo API

Замените координаты — получите те же графики для любой точки мира. Бесплатно, без регистрации.

In [ ]:
# ═══════════════════════════════════════════
#   ЗАМЕНИТЕ НА СВОЮ ТОЧКУ!
# ═══════════════════════════════════════════
LAT        = 56.86
LON        = 35.90
NAME       = 'Тверь'
YEAR_START = 1980
YEAR_END   = 2020
# ═══════════════════════════════════════════

import urllib.request, json

url = (
    f'https://archive-api.open-meteo.com/v1/archive?'
    f'latitude={LAT}&longitude={LON}'
    f'&start_date={YEAR_START}-01-01&end_date={YEAR_END}-12-31'
    f'&daily=temperature_2m_mean,temperature_2m_min,temperature_2m_max,'
    f'precipitation_sum,relative_humidity_2m_mean,wind_speed_10m_mean'
    f'&timezone=Europe%2FMoscow'
)

print(f'📡 Запрашиваем: {NAME} ({LAT}°N, {LON}°E)...')
try:
    with urllib.request.urlopen(url, timeout=30) as resp:
        data = json.loads(resp.read())
    df_api = pd.DataFrame(data['daily'])
    df_api['time'] = pd.to_datetime(df_api['time'])
    df_api = df_api.rename(columns={
        'time':'date','temperature_2m_mean':'Tmean',
        'temperature_2m_min':'Tmin','temperature_2m_max':'Tmax',
        'precipitation_sum':'Prcp',
        'relative_humidity_2m_mean':'RH','wind_speed_10m_mean':'Wind'
    }).set_index('date')
    df_api['month'] = df_api.index.month
    df_api['year']  = df_api.index.year
    n = df_api['year'].nunique()
    print(f'✅ {len(df_api):,} дней  |  T̄ = {df_api["Tmean"].mean():.1f}°C  |  P̄ = {df_api["Prcp"].sum()/n:.0f} мм/год')
    fname = f"{NAME.lower().replace(' ','_')}_{YEAR_START}_{YEAR_END}.csv"
    df_api.to_csv(fname)
    print(f'💾 Сохранено: {fname}')
except Exception as e:
    print(f'❌ Ошибка: {e}')

In [ ]:
# Быстрый климатический портрет скачанной точки
try:
    mon_api = df_api.groupby('month').agg(
        T=('Tmean','mean'),
        P=('Prcp', lambda x: x.sum()/df_api['year'].nunique())
    ).reset_index()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
    ax1.plot(mon_api['month'], mon_api['T'], 'o-', color='#e74c3c', lw=2.5, ms=8)
    ax1.axhline(0, color='gray', lw=1)
    ax1.set_xticks(range(1,13)); ax1.set_xticklabels(MONTHS_RU)
    ax1.set_title(f'Температура — {NAME}'); ax1.set_ylabel('°C')

    ax2.bar(mon_api['month'], mon_api['P'], color='#3498db', alpha=0.8)
    ax2.set_xticks(range(1,13)); ax2.set_xticklabels(MONTHS_RU)
    ax2.set_title(f'Осадки — {NAME}'); ax2.set_ylabel('мм/месяц')

    plt.suptitle(f'{NAME} | Open-Meteo (ERA5) | {YEAR_START}–{YEAR_END}', fontsize=13)
    plt.tight_layout()
    plt.show()
except NameError:
    print('Сначала запустите ячейку выше!')

---
## 📚 Полезные ресурсы

| Источник | Что даёт | Доступ |
|----------|----------|--------|
| **Open-Meteo** | ERA5 суточные, 1940–, без регистрации | archive-api.open-meteo.com |
| **Copernicus CDS** | ERA5 оригинал (NetCDF), любые переменные | cds.climate.copernicus.eu |
| **ВНИИГМИ-МЦД** | Российские метеостанции | meteo.ru |
| **NCEI GHCN-Daily** | Глобальная сеть станций | ncei.noaa.gov |
| **WorldClim / CHELSA** | Климатические нормы в растре (GeoTIFF) | worldclim.org |

---
*Ноутбук подготовлен для Школы молодых почвоведов — Цифровой трек, 2026*